[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/stevenelliottjr/healthcare-analytics-course-data/blob/main/notebooks/lab11_evaluating_ai.ipynb) · [Course home](https://colab.research.google.com/github/stevenelliottjr/healthcare-analytics-course-data/blob/main/notebooks/00_course_home.ipynb)

**Google Colab version.** **Before you start: File → Save a copy in Drive.** Work only in your copy (its title starts with *Copy of*); changes to this original are not saved. Rename your copy `Lab 11 — <your last name>` (click the title at the top left).

**To submit:** **Runtime → Run all** and check that every cell ran, then **File → Save**. Click
**Share** (top right), add **steven.elliott@student.fairfield.edu** as **Editor**, and send. Colab keeps your saved versions
under **File → Revision history**; **File → Save and pin revision** keeps one permanently, which is how you
mark a submission.

**API key.** In Colab this lab runs in **replay mode** (recorded examples) unless you add a key. For live mode, ask the instructor for one and add it in **Secrets** (key icon, left sidebar) as `ANTHROPIC_API_KEY`, with **Notebook access** on. Never paste a key into a cell.

**Coming back later?** Colab forgets variables and downloaded files when the runtime disconnects (after about 90 minutes idle). Open your copy and click **Runtime → Run all**.

# Lab 11 — Evaluating AI answers
*Week 11. Baseline: sections 1–5. Powerups: section 6.*

In week 10 you checked five answers by hand. That does not scale, and it does not tell you whether a change
to the prompt made the agent better or worse. An **eval set** does: a fixed list of questions with answer keys
computed by *your* code, a grader that scores the agent automatically, and a habit of reading every failure.
This week you write ten questions, grade the analyst on them, and sort its failures into categories.

**How labs work.** Save a copy in Drive, work in your copy, share it back with the instructor
when you submit. Every lab has a **baseline** (everyone finishes it in the session) and
**powerups** (optional depth). AI assistants may write code for you; **you own every line** and
may be asked to explain any of it. Cells marked `# TODO` are yours to fill in.

**No API key?** The agent replays four recorded example runs (clearly labelled) and marks your own questions
"not run". Everything else, including your keys and the graders, works the same. The milestone needs a live
score; in Colab, ask the instructor for a key and add it to Colab's **Secrets** (key icon, left sidebar) as `ANTHROPIC_API_KEY` with **Notebook access** on; lab 10, section 3 shows how. Without a key the notebook says `MODE = REPLAY`.

In [ ]:
# --- Course setup: run this cell first -------------------------------------------
# Loaders for the public datasets this course uses. The canonical copy of this code
# is labs/course_data.py in the course repo; you will read it line by line in week 3.
import io, json, urllib.request, urllib.parse
from pathlib import Path
import pandas as pd
import duckdb

CACHE = Path("data"); CACHE.mkdir(exist_ok=True)

def sql(query: str) -> pd.DataFrame:
    """Run SQL over any DataFrame in this notebook by its variable name (DuckDB)."""
    return duckdb.sql(query).df()

def _get(url, params=None, timeout=120) -> bytes:
    if params:
        url = url + "?" + urllib.parse.urlencode(params)
    req = urllib.request.Request(url, headers={"User-Agent": "healthcare-analytics-course/1.0"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()

# ---- CMS Care Compare (data.cms.gov/provider-data) --------------------------------
# File names change at every CMS refresh; dataset identifiers do not. We ask the
# catalog for the current CSV URL every time instead of hard-coding a file name.
CMS_CATALOG = "https://data.cms.gov/provider-data/api/1/metastore/schemas/dataset/items"
CMS_IDS = {
    "hospital_general_info": "xubh-q36u",   # one row per hospital
    "hrrp":                  "9n3s-kdb3",   # Hospital Readmissions Reduction Program, one row per hospital x measure
    "unplanned_visits":      "632h-zaca",   # readmissions / ED visits, one row per hospital x measure
    "complications_deaths":  "ynj2-r877",
    "timely_effective":      "yv7e-xc69",
}

def cms_csv_url(identifier: str) -> str:
    # Ask for the one dataset by id; the full keyword search can take minutes or time out.
    item = json.loads(_get(f"{CMS_CATALOG}/{identifier}", timeout=30))
    for d in item.get("distribution", []):
        if d.get("mediaType") == "text/csv":
            return d["downloadURL"]
    raise KeyError(f"No CSV distribution found for CMS dataset {identifier}")

# Course mirror: a snapshot of every CMS file the labs use, for when CMS is down or rate-limits
# a classroom downloading the same file at once. The live source is always tried first.
COURSE_DATA_BASE = "https://raw.githubusercontent.com/stevenelliottjr/healthcare-analytics-course-data/main/data"

def _fetch_or_mirror(path: Path, fetch, mirror_file: str) -> None:
    """Save `path` from the live source; if that fails, from the course mirror."""
    try:
        path.write_bytes(fetch())
    except Exception as e:
        print(f"Live download failed ({type(e).__name__}); using the course mirror for {path.name}.")
        path.write_bytes(_get(f"{COURSE_DATA_BASE}/{mirror_file}", timeout=300))

def load_cms(name: str, **read_csv_kwargs) -> pd.DataFrame:
    """Download (once) and read a Care Compare file. Everything is read as TEXT on purpose:
    IDs keep their leading zeros and 'Not Available' stays visible instead of silently
    becoming NaN. Converting columns to numbers is your job, and part of the lesson."""
    path = CACHE / f"cms_{name}.csv"
    if not path.exists():
        _fetch_or_mirror(path, lambda: _get(cms_csv_url(CMS_IDS[name]), timeout=300), f"cms/cms_{name}.csv")
    kw = dict(dtype=str, keep_default_na=False, encoding="utf-8-sig")
    kw.update(read_csv_kwargs)
    return pd.read_csv(path, **kw)

def to_num(s: pd.Series) -> pd.Series:
    """Text -> number; anything that is not a number ('Not Available', 'N/A', '') becomes NaN."""
    return pd.to_numeric(s, errors="coerce")

# ---- CDC PLACES, county release (Socrata API on data.cdc.gov) ---------------------
PLACES_COUNTY = "https://data.cdc.gov/resource/swc5-untb.json"
PLACES_SELECT = ("year,stateabbr,statedesc,locationname,locationid,categoryid,measureid,measure,"
                 "data_value,low_confidence_limit,high_confidence_limit,totalpopulation")

def load_places(measure_ids, data_value_type="Age-adjusted prevalence", state=None, page=50000) -> pd.DataFrame:
    """County-level model-based prevalence estimates (percent of adults). locationid is the
    5-digit county FIPS. Note: Connecticut reports *planning regions*, not counties, since 2022."""
    where = "measureid in(" + ",".join(f"'{m}'" for m in measure_ids) + f") AND data_value_type='{data_value_type}'"
    if state:
        where += f" AND stateabbr='{state}'"
    frames, offset = [], 0
    while True:
        chunk = pd.DataFrame(json.loads(_get(PLACES_COUNTY, {"$where": where, "$select": PLACES_SELECT,
                                                               "$limit": page, "$offset": offset})))
        frames.append(chunk)
        if len(chunk) < page:
            break
        offset += page
    df = pd.concat(frames, ignore_index=True)
    for c in ["data_value", "low_confidence_limit", "high_confidence_limit", "totalpopulation"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

# ---- Week 4 powerups (links checked 2026-10-26; 2022 is the latest SVI release) ----
SVI_COUNTY_URL = "https://svi.cdc.gov/Documents/Data/2022/csv/states_counties/SVI_2022_US_county.csv"
def load_svi_county() -> pd.DataFrame:
    """CDC/ATSDR Social Vulnerability Index 2022, one row per county.
    Key columns: FIPS, STATE, COUNTY, E_TOTPOP, RPL_THEMES (overall percentile 0-1),
    RPL_THEME1..RPL_THEME4. The value -999 means 'not available'."""
    path = CACHE / "svi_2022_county.csv"
    if not path.exists():
        path.write_bytes(_get(SVI_COUNTY_URL, timeout=300))
    return pd.read_csv(path, dtype={"FIPS": str}).replace(-999, pd.NA)

CHR_URL = "https://www.countyhealthrankings.org/sites/default/files/media/document/analytic_data2025_v3.csv"
def load_county_health_rankings() -> pd.DataFrame:
    """County Health Rankings analytic file. Row 0 holds readable labels and row 1 the variable
    codes; we skip the labels so the codes (fipscode, v001..v1xx) become the column names.
    The data dictionary on the CHR site maps each v-code to a measure."""
    path = CACHE / "chr_analytic_2025_v3.csv"
    if not path.exists():
        path.write_bytes(_get(CHR_URL, timeout=300))
    return pd.read_csv(path, skiprows=[0], dtype={"fipscode": str, "statecode": str, "countycode": str})

print("course setup ready")


In [ ]:
# --- Week 11 packages: anthropic ------------------------------------------------
# Not every base image has them (Hex does not). Installs once per kernel session.
import importlib.util, shutil, subprocess, sys

_need = [pkg for pkg, mod in [("anthropic", "anthropic")]
         if importlib.util.find_spec(mod) is None]
if _need:
    if shutil.which("uv"):  # Hex: the kernel's virtualenv has no pip module, but uv is on the PATH
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_need], check=True)
    else:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_need], check=True)
    importlib.invalidate_caches()
print("week 11 packages ready")


## 1. The analyst, again
The next four cells are lab 10's table, tool, agent and recordings, unchanged except for one addition: for
grading, `EVAL_SYSTEM` asks the agent to end every reply with a line `FINAL: <answer>`. Graders read that line
and nothing else. (Why not grade the whole reply? Because "2,621 of 3,055 rows" contains two numbers, and a
grader that picks the wrong one is a grader you cannot trust.)

**Baseline:** run them and note the mode (`LIVE` or `REPLAY`). Nothing else to report.

In [ ]:
# Three CMS files, read as text, turned into one tidy table the agent can query.
hrrp = load_cms("hrrp")
hgi = load_cms("hospital_general_info")
uhv = load_cms("unplanned_visits")

hf = hrrp[hrrp["Measure Name"].eq("READM-30-HF-HRRP")]
hospitals = pd.DataFrame({
    "facility_id":     hf["Facility ID"],
    "facility_name":   hf["Facility Name"],
    "state":           hf["State"],
    "hf_discharges":   to_num(hf["Number of Discharges"]),
    "hf_readmissions": to_num(hf["Number of Readmissions"]),   # 'Too Few to Report' -> missing, not 0
    "excess_ratio":    to_num(hf["Excess Readmission Ratio"]),
    "predicted_rate":  to_num(hf["Predicted Readmission Rate"]),
    "expected_rate":   to_num(hf["Expected Readmission Rate"]),
})
info = hgi[["Facility ID", "City/Town", "County/Parish", "Hospital Type", "Hospital Ownership",
            "Emergency Services", "Hospital overall rating"]].copy()
info.columns = ["facility_id", "city", "county", "hospital_type", "ownership", "emergency_services", "star_rating"]
hospitals = hospitals.merge(info, on="facility_id", how="left")
hospitals["star_rating"] = to_num(hospitals["star_rating"])

rate = uhv[uhv["Measure ID"].eq("READM_30_HF")]
rate = pd.DataFrame({"facility_id": rate["Facility ID"], "hf_readm_rate": to_num(rate["Score"])})
hospitals = hospitals.merge(rate, on="facility_id", how="left")

# 1 = in the penalty range, 0 = not, missing = no ratio published (never 0).
hospitals["penalized"] = (hospitals["excess_ratio"] > 1).astype(float).where(hospitals["excess_ratio"].notna())
hospitals = hospitals.reset_index(drop=True)
print(hospitals.shape, "|", int(hospitals["excess_ratio"].notna().sum()), "hospitals with an excess readmission ratio")


In [ ]:
import re

# What the agent is told about the table. It cannot see the data until it queries it,
# so this text is all it knows. Every claim in it should be true; you check that in section 1.
SCHEMA = """Table: hospitals. One row per hospital in Medicare's Hospital Readmissions Reduction Program
(HRRP) for heart failure, discharges July 2021 - June 2024; about 3,000 rows, all acute care hospitals.
Missing values are NULL.

facility_id         VARCHAR  CMS certification number (6 characters; text, keep leading zeros)
facility_name       VARCHAR  hospital name, upper case
state               VARCHAR  2-letter postal code (includes DC and territories such as PR)
city, county        VARCHAR  from Hospital General Information (NULL for ~20 hospitals not in that file)
hospital_type       VARCHAR  from Hospital General Information
ownership           VARCHAR  e.g. 'Proprietary', 'Voluntary non-profit - Private',
                             'Voluntary non-profit - Church', 'Government - Local'
emergency_services  VARCHAR  'Yes' or 'No'
star_rating         DOUBLE   CMS overall star rating 1-5; NULL if not rated
hf_discharges       DOUBLE   heart-failure discharges in the HRRP period; NULL if not reported
hf_readmissions     DOUBLE   heart-failure readmissions; NULL if 'Too Few to Report' or not available
excess_ratio        DOUBLE   HRRP excess readmission ratio = predicted / expected; NULL if not published
predicted_rate      DOUBLE   predicted readmission rate, percent
expected_rate       DOUBLE   expected readmission rate, percent
hf_readm_rate       DOUBLE   risk-standardized 30-day heart-failure readmission rate, percent
                             (Care Compare, July 2023 - June 2025)
penalized           DOUBLE   1 if excess_ratio > 1 (the HRRP penalty range), 0 if not, NULL if no ratio

Not in this table: hospital-wide readmission rates, other conditions, urban/rural flags,
patient-level data, payments or penalty amounts."""

QUERY_TOOL = {
    "name": "query_hospitals",
    "description": ("Run one read-only DuckDB SELECT against the `hospitals` table and get the rows back "
                    "as JSON (at most 100 rows). Use it for every number you report."),
    "strict": True,
    "input_schema": {
        "type": "object",
        "properties": {"sql": {"type": "string", "description": "A single SELECT (or WITH ... SELECT) statement."}},
        "required": ["sql"],
        "additionalProperties": False,
    },
}

# The tool gets its own DuckDB connection that can see ONE table and no files.
# (The course sql() helper can see every DataFrame in the notebook; the agent should not.)
_agent_db = duckdb.connect(config={"enable_external_access": False})
_agent_db.register("hospitals", hospitals)
_FORBIDDEN = re.compile(r"\b(insert|update|delete|drop|alter|create|attach|detach|copy|pragma|install|load|export|import|call|set)\b", re.I)

def query_hospitals(sql: str, max_rows: int = 100) -> dict:
    """The agent's only tool: run a SELECT, return rows as plain Python. Refuses anything else."""
    q = sql.strip().rstrip(";").strip()
    if ";" in q:
        raise ValueError("one statement at a time")
    if not re.match(r"(?is)^(select|with)\b", q):
        raise ValueError("only SELECT queries are allowed")
    if _FORBIDDEN.search(q):
        raise ValueError(f"keyword not allowed: {_FORBIDDEN.search(q).group(0)}")
    df = _agent_db.execute(q).df()
    return {"row_count": len(df), "truncated": len(df) > max_rows, "columns": list(df.columns),
            "rows": json.loads(df.head(max_rows).to_json(orient="records"))}


In [ ]:
import os, time, textwrap
import anthropic

MODEL = "claude-opus-5-5"
# US dollars per million tokens for this model (Anthropic list price, October 2026).
PRICE_PER_MTOK = {"input": 4.00, "output": 20.00, "cache_read": 0.20, "cache_write": 5.00}
MAX_TURNS = 8   # model calls per question; a runaway loop stops here

# Where the key comes from, tried in order:
#   1. Hex: a workspace secret named ANTHROPIC_API_KEY appears as a Python variable of that name.
#   2. Colab: the Secrets panel (key icon, left sidebar), secret ANTHROPIC_API_KEY, notebook access on.
#   3. Anywhere else (a laptop): an environment variable.
# No key found -> replay mode. In Colab that is the default.
_key = globals().get("ANTHROPIC_API_KEY")
if not (isinstance(_key, str) and _key.strip()):
    try:
        from google.colab import userdata
        _key = userdata.get("ANTHROPIC_API_KEY")
    except Exception:   # not in Colab, secret not set, or notebook access not granted
        _key = None
if not (isinstance(_key, str) and _key.strip()):
    _key = os.environ.get("ANTHROPIC_API_KEY")
API_KEY = _key.strip() if isinstance(_key, str) and _key.strip() else None
MODE = "live" if API_KEY else "replay"
client = anthropic.Anthropic(api_key=API_KEY) if API_KEY else None
print(f"MODE = {MODE.upper()}  (model {MODEL}, anthropic SDK {anthropic.__version__})")
if MODE == "replay":
    print("No ANTHROPIC_API_KEY found, so the agent will REPLAY recorded example transcripts. "
          "The SQL in them runs for real; the agent's words are a recording, not a live model.")

SYSTEM = f"""You are a careful healthcare data analyst. Answer questions about U.S. hospitals by calling
the query_hospitals tool; never state a number you did not get from a query. Say which filter and which
denominator you used. If the table cannot answer the question, say so plainly and offer the closest thing
it can answer. Keep the final answer short.

{SCHEMA}"""

def run_agent(question: str, system: str = SYSTEM) -> dict:
    """Ask the model; run every tool call it makes; repeat until it stops asking for tools."""
    run = {"question": question, "mode": "live", "model": MODEL, "events": [], "usage": [], "answer": "", "stop": None}
    messages = [{"role": "user", "content": question}]
    for _ in range(MAX_TURNS):
        t0 = time.perf_counter()
        try:
            response = client.beta.messages.create(
                model=MODEL, max_tokens=16000, system=system, tools=[QUERY_TOOL], messages=messages,
                output_config={"effort": "medium"},
                # If a safety filter declines a request, retry it on Anthropic's recommended fallback model.
                betas=["server-side-fallback-2026-07-01"], fallbacks="default",
            )
        except anthropic.AuthenticationError:
            run["events"].append({"kind": "error", "text": "API key rejected. Ask the instructor."}); break
        except anthropic.RateLimitError:
            run["events"].append({"kind": "error", "text": "Rate limited. Wait a minute and rerun."}); break
        except anthropic.APIStatusError as e:
            run["events"].append({"kind": "error", "text": f"API error {e.status_code}: {e.message}"}); break
        except anthropic.APIConnectionError:
            run["events"].append({"kind": "error", "text": "Could not reach the API."}); break
        u = response.usage
        run["usage"].append({"input": u.input_tokens, "output": u.output_tokens,
                             "cache_read": u.cache_read_input_tokens or 0,
                             "cache_write": u.cache_creation_input_tokens or 0,
                             "seconds": round(time.perf_counter() - t0, 1)})
        run["stop"] = response.stop_reason
        # Send the assistant turn back exactly as received (it includes hidden thinking blocks).
        messages.append({"role": "assistant", "content": response.content})
        for block in response.content:
            if block.type == "text" and block.text.strip():
                run["events"].append({"kind": "text", "text": block.text})
        if response.stop_reason == "refusal":
            run["events"].append({"kind": "error", "text": "The model declined this request."}); break
        if response.stop_reason != "tool_use":
            break
        results = []
        for block in response.content:
            if block.type != "tool_use":
                continue
            sql_text = block.input.get("sql", "")
            run["events"].append({"kind": "tool_call", "tool": block.name, "sql": sql_text})
            try:
                if block.name != "query_hospitals":
                    raise ValueError(f"unknown tool {block.name}")
                out = query_hospitals(sql_text)
                content, err = json.dumps(out, default=str), None
            except Exception as e:  # the error goes back to the model, which can fix its SQL
                out, err = None, f"{type(e).__name__}: {e}"
                content = "ERROR: " + err
            run["events"].append({"kind": "tool_result", "result": out, "error": err})
            results.append({"type": "tool_result", "tool_use_id": block.id, "content": content, "is_error": err is not None})
        messages.append({"role": "user", "content": results})   # all results in one message
    texts = [e["text"] for e in run["events"] if e["kind"] == "text"]
    run["answer"] = texts[-1] if texts else ""
    return run

def run_cost(run: dict) -> float:
    """Dollars for one run, from the token counts the API reported."""
    return sum(u[k] * PRICE_PER_MTOK[k] for u in run["usage"] for k in PRICE_PER_MTOK) / 1e6

def show(run: dict, max_rows: int = 8):
    """Print a run as a transcript: what the agent said, each tool call, and what came back."""
    print("=" * 96)
    tag = "LIVE" if run["mode"] == "live" else "REPLAY (recorded example, not a live model)"
    print(f"[{tag}]  Q: {run['question']}")
    for e in run["events"]:
        if e["kind"] == "text":
            print("\nAGENT: " + e["text"].strip())
        elif e["kind"] == "tool_call":
            print(f"\n  -> {e['tool']}(sql=)\n" + textwrap.indent(e["sql"].strip(), "     "))
        elif e["kind"] == "tool_result":
            if e["error"]:
                print("  <- ERROR " + e["error"])
            else:
                r = e["result"]
                print(f"  <- {r['row_count']} row(s)" + (" (truncated)" if r.get("truncated") else ""))
                if r["rows"]:
                    print(textwrap.indent(pd.DataFrame(r["rows"]).head(max_rows).to_string(index=False), "     "))
        elif e["kind"] == "error":
            print("\n!! " + e["text"])
    if run["usage"]:
        tin = sum(u["input"] for u in run["usage"]); tout = sum(u["output"] for u in run["usage"])
        print(f"\n[{len(run['usage'])} model call(s), {tin:,} input + {tout:,} output tokens, ${run_cost(run):.4f}]")
    else:
        print("\n[replay: no model calls, $0]")

# For grading, the agent must end with a machine-readable line.
EVAL_SYSTEM = SYSTEM + """

End every reply with one final line of the form `FINAL: <answer>`, where <answer> is a bare number (no
units, no % sign, no thousands separators), or a name exactly as it appears in the table, or
`FINAL: NOT AVAILABLE` if the table cannot answer the question."""


In [ ]:
# Recorded example runs for REPLAY mode (written by course staff, 2026-10-26). The SQL runs for real;
# the agent's words are a recording, not a live model. One of the four is wrong on purpose.
EXAMPLES_Q = {
    "E1": "How many hospitals in the table have a heart-failure excess readmission ratio?",
    "E2": "Which Connecticut hospital has the highest heart-failure excess readmission ratio?",
    "E3": "What percent of Florida hospitals with an excess readmission ratio are in the penalty range?",
    "E4": "What is the average hospital-wide readmission rate for Connecticut hospitals?",
}
RECORDED = {
    EXAMPLES_Q["E1"]: {"steps": [("Counting non-missing ratios.",
                                  "SELECT COUNT(excess_ratio) AS n FROM hospitals")],
                       "answer": "2,621 hospitals have a published excess readmission ratio.\nFINAL: 2621"},
    EXAMPLES_Q["E2"]: {"steps": [("Sorting Connecticut hospitals by ratio.",
                                  "SELECT facility_name, excess_ratio FROM hospitals\nWHERE state = 'CT' AND excess_ratio IS NOT NULL\n"
                                  "ORDER BY excess_ratio DESC LIMIT 3")],
                       "answer": "Hartford Hospital, with a ratio of 1.1395.\nFINAL: HARTFORD HOSPITAL"},
    EXAMPLES_Q["E3"]: {"steps": [("Share of Florida hospitals with a ratio above 1.",
                                  "SELECT ROUND(100.0 * SUM(CASE WHEN excess_ratio > 1 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct\n"
                                  "FROM hospitals WHERE state = 'FL'")],
                       "answer": "63.9% of Florida hospitals are in the penalty range.\nFINAL: 63.9"},
    EXAMPLES_Q["E4"]: {"steps": [("Checking what the table has; there is no hospital-wide measure.",
                                  "SELECT COUNT(hf_readm_rate) AS n_hf_rates FROM hospitals WHERE state = 'CT'")],
                       "answer": ("The table has no hospital-wide readmission rate, only heart failure.\n"
                                  "FINAL: NOT AVAILABLE")},
}

def replay_agent(question: str) -> dict:
    """Replay a recorded transcript. The SQL runs now, against your table; the words are the recording."""
    run = {"question": question, "mode": "replay", "model": MODEL + " (recorded)", "events": [], "usage": [],
           "answer": "", "stop": "end_turn"}
    rec = RECORDED.get(question)
    if rec is None:
        run["events"].append({"kind": "error", "text": "No recording for this question. Replay mode only knows "
                              "the example questions; live mode needs an API key (set up in lab 10, section 3)."})
        return run
    for narration, q in rec["steps"]:
        run["events"].append({"kind": "text", "text": narration})
        run["events"].append({"kind": "tool_call", "tool": "query_hospitals", "sql": q})
        try:
            run["events"].append({"kind": "tool_result", "result": query_hospitals(q), "error": None})
        except Exception as e:
            run["events"].append({"kind": "tool_result", "result": None, "error": f"{type(e).__name__}: {e}"})
    run["events"].append({"kind": "text", "text": rec["answer"]})
    run["answer"] = rec["answer"]
    return run

def ask(question: str, **kw) -> dict:
    return run_agent(question, **kw) if MODE == "live" else replay_agent(question)


## 2. The harness
An eval item is a question, a **key query** you wrote, a **kind**, and a **tolerance**:

- `number`: the agent's FINAL number must be within `tolerance` of your key. Pick the tolerance from the
  question: a count needs 0; a percent rounded to one decimal needs 0.1. A tolerance is a statement about what
  "right" means, so write it down before you see the agent's answer.
- `exact`: the FINAL text must match your key, ignoring case and spaces. Good for "which hospital / which state".
- `unanswerable`: the right answer is `NOT AVAILABLE`. Every eval set needs these; an agent that never says
  "I can't" will make something up.

`compute_expected` runs your key query and stores the value, so a key is never typed by hand.

**Baseline:** read the harness, run the grader tests, and add **two tests of your own** to the test cell: one
answer that should pass and one that should fail but that a careless grader might pass.

In [ ]:
from dataclasses import dataclass
import re

@dataclass
class EvalItem:
    id: str
    question: str
    kind: str                # "number", "exact" or "unanswerable"
    key_sql: str = ""        # YOUR query that computes the answer (empty for "unanswerable")
    tolerance: float = 0.0   # "number" only: how far off still counts as right (same units as the answer)
    notes: str = ""          # what this question tests
    expected: object = None  # filled in by compute_expected(); never typed by hand

def compute_expected(item: EvalItem):
    """Run the key query. It must return exactly one value."""
    if item.kind == "unanswerable":
        item.expected = "NOT AVAILABLE"
        return item.expected
    df = sql(item.key_sql)
    if df.shape != (1, 1):
        raise ValueError(f"{item.id}: key_sql must return one row and one column, got {df.shape}")
    value = df.iat[0, 0]
    item.expected = float(value) if item.kind == "number" else str(value)
    return item.expected

def parse_final(answer: str):
    """The text after the last 'FINAL:' in the agent's reply, or None if there is none."""
    found = re.findall(r"FINAL:\s*(.+)", answer or "")
    return found[-1].strip().strip("`*").strip() if found else None

def _norm(s) -> str:
    return re.sub(r"\s+", " ", str(s)).strip().strip(".").casefold()

def grade_number(final, expected, tolerance) -> bool:
    if final is None:
        return False
    nums = re.findall(r"-?\d+(?:\.\d+)?", final.replace(",", ""))
    return bool(nums) and abs(float(nums[0]) - float(expected)) <= tolerance + 1e-9

def grade_exact(final, expected) -> bool:
    return final is not None and _norm(final) == _norm(expected)

def grade_unanswerable(final) -> bool:
    return final is not None and _norm(final) == "not available"

def grade(item: EvalItem, answer: str) -> dict:
    final = parse_final(answer)
    if item.kind == "number":
        ok = grade_number(final, item.expected, item.tolerance)
    elif item.kind == "exact":
        ok = grade_exact(final, item.expected)
    elif item.kind == "unanswerable":
        ok = grade_unanswerable(final)
    else:
        raise ValueError(f"{item.id}: unknown kind {item.kind!r}")
    return {"final": final, "passed": ok}

def run_eval(items: list) -> pd.DataFrame:
    """Ask the agent every question, grade each answer, return one row per question."""
    rows = []
    for it in items:
        if it.expected is None:
            compute_expected(it)
        run = ask(it.question, system=EVAL_SYSTEM)
        ran = bool(run["answer"])
        g = grade(it, run["answer"]) if ran else {"final": None, "passed": None}
        rows.append({"id": it.id, "kind": it.kind, "question": it.question, "expected": it.expected,
                     "final": g["final"], "passed": g["passed"], "status": "graded" if ran else "not run",
                     "usd": round(run_cost(run), 4), "answer": run["answer"], "run": run})
    return pd.DataFrame(rows)

def scorecard(results: pd.DataFrame) -> pd.DataFrame:
    graded = results[results["status"].eq("graded")].astype({"passed": bool})
    by_kind = graded.groupby("kind")["passed"].agg(passed="sum", graded="size")
    by_kind.loc["ALL"] = [int(graded["passed"].sum()), len(graded)]
    by_kind = by_kind.astype(int)
    by_kind["score"] = (by_kind["passed"] / by_kind["graded"].where(by_kind["graded"] > 0)).round(2)
    return by_kind


In [ ]:
# A grader you have not tested is another thing to verify. These must all pass.
assert parse_final("blah\nFINAL: 48.9") == "48.9"
assert parse_final("no final line") is None
assert grade_number("48.9%", 48.9, 0.1) and grade_number("2,621", 2621, 0)
assert not grade_number("49.1", 48.9, 0.1)
assert grade_exact("Hartford Hospital", "HARTFORD HOSPITAL")
assert grade_unanswerable("NOT AVAILABLE") and not grade_unanswerable("22.01")

# TODO: two tests of your own (one should pass, one should fail but looks close).
print("grader tests pass")


## 3. Write ten questions with keys
**Baseline:** ten `EvalItem`s in `EVAL`, written by you, about the `hospitals` table. Rules:
- at least 3 `number`, 2 `exact`, 2 `unanswerable`;
- at least one question where the **denominator** matters (missing values), one **ranking**, one **filter**
  on a column other than `state`;
- every key query returns one value; run it and look at the value before you accept it;
- phrase questions the way a manager would ask them, not the way SQL would.

Four worked examples come first; they are not part of your ten.

In [ ]:
# The four worked examples (recorded in replay mode). Read them, then write your own ten below.
EXAMPLES = [
    EvalItem("E1", EXAMPLES_Q["E1"], "number", "SELECT COUNT(excess_ratio) FROM hospitals", 0,
             "count that must skip missing ratios"),
    EvalItem("E2", EXAMPLES_Q["E2"], "exact",
             "SELECT facility_name FROM hospitals WHERE state = 'CT' ORDER BY excess_ratio DESC NULLS LAST LIMIT 1",
             notes="ranking; NULLs must not sort first"),
    EvalItem("E3", EXAMPLES_Q["E3"], "number",
             "SELECT ROUND(100 * AVG(penalized), 1) FROM hospitals WHERE state = 'FL'", 0.1,
             "denominator: hospitals with a ratio, not all rows"),
    EvalItem("E4", EXAMPLES_Q["E4"], "unanswerable", notes="data the table does not have"),
]
for _it in EXAMPLES:
    compute_expected(_it)
pd.DataFrame([{"id": it.id, "kind": it.kind, "expected": it.expected, "tolerance": it.tolerance} for it in EXAMPLES])


In [ ]:
# TODO: your ten. Mix the kinds (at least 3 number, 2 exact, 2 unanswerable) and write the key_sql yourself.
EVAL = [
    # EvalItem("Q1", "question in plain English", "number", "SELECT ... one value ...", 0.1, "what it tests"),
]
for _it in EVAL:
    compute_expected(_it)
pd.DataFrame([{"id": it.id, "kind": it.kind, "expected": it.expected, "tolerance": it.tolerance, "notes": it.notes}
              for it in EVAL])


## 4. Run the agent and score it
`run_eval` asks every question, grades every answer, and records cost. The scorecard counts only graded
questions. In replay mode your ten show `not run`.

**Baseline:** report the score on **your ten** (live) and on the examples, the total cost, and one sentence on
whether the score by kind tells you something the overall score hides.

In [ ]:
results = run_eval(EXAMPLES + EVAL)
print(f"mode {MODE} | cost ${results['usd'].sum():.3f}")
print("examples:"); print(scorecard(results[results["id"].str.startswith("E")]))
print("\nyour ten:"); print(scorecard(results[~results["id"].str.startswith("E")]))
results[["id", "kind", "expected", "final", "passed", "status"]]


## 5. Read every failure
A score is the start of the analysis. For each failed question, read the transcript (`show(...)`), decide
**why** it failed, and label it with one of `FAILURE_CATEGORIES`. Check your own key first: sometimes the
agent is right and the key or the question is wrong.

**Baseline:** a label for every failure, the table below, and three sentences: which category is most common,
whether any failure was really your fault (key or question), and what one change to the agent (prompt, schema
text, tool) you would try first.

In [ ]:
FAILURE_CATEGORIES = {
    "denominator":  "right numerator, wrong base (counted rows with missing values, or the wrong group)",
    "filter":       "wrong or missing WHERE: dropped a group, wrong state, wrong label",
    "column":       "used a different column or measure than the question asked for",
    "made_up":      "answered a question the data cannot answer",
    "over_refused": "said NOT AVAILABLE when the table could answer",
    "format":       "right answer, but the FINAL line could not be graded",
    "key_or_question": "your key or your question was wrong or ambiguous (it happens: fix it and rerun)",
    "other":        "anything else (explain in the table below)",
}

# TODO: one label per failed question, e.g. {"E3": "denominator"}. Read the transcript first: show(results.set_index("id").loc["E3", "run"])
labels = {}

failed = results[results["passed"].eq(False)]
failed.assign(category=failed["id"].map(labels))[["id", "question", "expected", "final", "category"]]


**Failures (edit this cell):**

| id | Expected | Agent's FINAL | Category | What happened |
|---|---|---|---|---|
| | | | | |

**Three sentences:** *(edit this cell)*

## Milestone — Evaluation set (due with this lab)
In your capstone notebook: **ten questions with answer keys** about *your* capstone data, and **the AI analyst's
score on them**. To point the analyst at your data, replace the `hospitals` table and the `SCHEMA` text in the
Table and Tool cells with your own table and an honest description of it (every column, its units, what is
missing, and what is *not* in the table). Keep the harness unchanged. Submit the scorecard, the failure table,
and the three sentences from section 5.

## 6. Powerup: an LLM-graded eval
Programmatic graders are strict and blind: "sixty-seven percent" fails a number grader, and a correct number
with a wrong explanation passes. An **LLM judge** reads the whole reply. Build one: send the question, the key and
the agent's reply to the model, ask for a verdict in a fixed JSON shape, and compare its verdicts with
`grade()` on every graded question. Report the agreement rate and read every disagreement: who was right?

Then try to fool it: write one reply that is wrong but persuasive and see whether the judge passes it. That is
why the course's milestone uses keys computed by your own code, with the judge as a second opinion.

In [ ]:
# TODO (powerup): an LLM grader. Ask the model whether each answer matches the key, with a fixed JSON reply,
# then compare its verdicts with grade(). Structured output keeps the reply parseable:
JUDGE_SCHEMA = {"type": "object",
                "properties": {"verdict": {"type": "string", "enum": ["correct", "incorrect"]},
                               "reason": {"type": "string"}},
                "required": ["verdict", "reason"], "additionalProperties": False}
# def llm_judge(item, answer) -> dict:
#     response = client.beta.messages.create(model=MODEL, max_tokens=4000, messages=[...],
#         output_config={"effort": "low", "format": {"type": "json_schema", "schema": JUDGE_SCHEMA}},
#         betas=["server-side-fallback-2026-07-01"], fallbacks="default")
#     return json.loads(next(b.text for b in response.content if b.type == "text"))
